In [6]:
%cd /home/parthgandhi/Projects/MLBot/mlbot

/home/parthgandhi/Projects/MLBot/mlbot


In [7]:
import polars as pl
from src.swing_model.features import basic_features, moving_average_features

In [10]:
data = pl.scan_parquet("src/test_data.parquet")

In [14]:
# res = basic_features(data=data)
# res = moving_average_features(data=res)

In [15]:
# res.filter(pl.col("symbol").is_in(["ITC"])).remove(
#     pl.any_horizontal(pl.col("*").is_null())
# ).collect()

In [36]:
(
    data.with_columns(pl.col("timestamp").cast(pl.Date()))
    .drop("volume")
    .with_columns(
        # Prev Price Columns
        [
            pl.col(col)
            .shift(1)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .alias(f"prev_{col}")
            for col in ["open", "high", "low", "close"]
        ]
        # Close SMA expression
        + [
            pl.col(col)
            .rolling_mean(window_size=n)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(2)
            .alias(f"{col}_sma_{n}")
            for n in [50, 200]
            for col in ["close"]
        ]
        # Close EMA Experssion
        + [
            pl.col("close")
            .ewm_mean(alpha=2 / (n + 1))
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(2)
            .alias(f"close_ema_{n}")
            for n in [9, 21]
        ]
        # 52 week high
        + [
            pl.col("close")
            .rolling_max(window_size=252)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .alias("high_52W")
        ]
        # 52 week low
        + [
            pl.col("close")
            .rolling_min(window_size=252)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .alias("low_52W")
        ]
        # Candle Red or Green
        + [
            pl.when(pl.col("close") > pl.col("open"))
            .then(True)
            .otherwise(False)
            .alias("is_green_candle")
        ]
        # Range of Candle
        + [(pl.col("high") - pl.col("low")).round(2).alias("range")]
        # Body of Canlde
        + [(pl.col("open") - pl.col("close")).abs().round(2).alias("body")]
        # Upper Wick
        + [
            (pl.col("high") - pl.max_horizontal("open", "close"))
            .round(2)
            .alias("upper_wick")
        ]
        # Lower Wick
        + [
            (pl.min_horizontal("open", "close") - pl.col("low"))
            .round(2)
            .alias("lower_wick")
        ]
    )
    .with_columns(
        # Log Returns Calculated for Changes
        [
            (pl.col(col) / pl.col(f"prev_{col}"))
            .log()
            .round(4)
            .alias(f"log_return_{col}")
            for col in ["open", "high", "low", "close"]
        ]
        # Percentage off 52 week high
        + [
            ((pl.col("close") / pl.col("high_52W")) - 1)
            .clip(upper_bound=1)
            .round(4)
            .alias("dst_from_high_52W")
        ]
        # True Range Calculation for ATR
        + [
            pl.max_horizontal(
                pl.col("high") - pl.col("low"),
                (pl.col("high") - pl.col("prev_close")).abs(),
                (pl.col("low") - pl.col("prev_close")).abs(),
            ).alias("true_range")
        ]
        # Calculate Standard Deviation based on Close, EMA9 and EMA21
        + [
            pl.concat_list("close_ema_9", "close_ema_21", "close")
            .list.std(ddof=0)
            .round(4)
            .alias("std_9_21")
        ]
        # Calculate Standard Deviation based on Close, EMA9, EMA21 and SMA50
        + [
            pl.concat_list("close_ema_9", "close_ema_21", "close", "close_sma_50")
            .list.std(ddof=0)
            .round(4)
            .alias("std_9_21_50")
        ]
        # Body, Upper Wick and Lower Wick Pct to Range
        + [
            (pl.col(col) / pl.col("range")).round(4).alias(f"{col}_to_range_pct")
            for col in ["body", "upper_wick", "lower_wick"]
        ]
        # Distance Between SMA 50 and SMA 200
        + [
            ((pl.col(col) / pl.col("close_sma_200")) - 1)
            .round(4)
            .alias(f"{col}_dist_from_close_sma_200")
            for col in ["close", "close_sma_50"]
        ]
        # SMA50 GTE SMA200
        + [
            pl.when(pl.col("close_sma_50") >= pl.col("close_sma_200"))
            .then(True)
            .otherwise(False)
            .alias("is_sma_50_gte_sma_200")
        ]
        # Distance of Close and Low to Moving Averages
        + [
            (((pl.col(m_col) / pl.col(c_col)) - 1).round(4)).alias(
                f"{m_col}_dist_from_{c_col}"
            )
            for m_col in ["close", "low"]
            for c_col in ["close_ema_9", "close_ema_21", "close_sma_50"]
        ]
    )
    .with_columns(
        # Calculate ATR using True Range
        [
            (pl.col("true_range") / pl.col("close"))
            .ewm_mean(alpha=2 / (n + 1))
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(4)
            .alias(f"atr_pct_{n}")
            for n in [20]
        ]
        # Calculate the smooth dispersion score based on the normalzied standard deviation
        + [
            pl.col(col)
            .rolling_mean(window_size=n)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(4)
            .alias(f"{col}_dispersion_{n}")
            for n in [5, 10, 15, 20]
            for col in ["std_9_21", "std_9_21_50"]
        ]
    )
).collect()

symbol,timestamp,open,high,low,close,prev_open,prev_high,prev_low,prev_close,close_sma_50,close_sma_200,close_ema_9,close_ema_21,high_52W,low_52W,is_green_candle,range,body,upper_wick,lower_wick,log_return_open,log_return_high,log_return_low,log_return_close,dst_from_high_52W,true_range,std_9_21,std_9_21_50,body_to_range_pct,upper_wick_to_range_pct,lower_wick_to_range_pct,close_dist_from_close_sma_200,close_sma_50_dist_from_close_sma_200,is_sma_50_gte_sma_200,close_dist_from_close_ema_9,close_dist_from_close_ema_21,close_dist_from_close_sma_50,low_dist_from_close_ema_9,low_dist_from_close_ema_21,low_dist_from_close_sma_50,atr_pct_20,std_9_21_dispersion_5,std_9_21_50_dispersion_5,std_9_21_dispersion_10,std_9_21_50_dispersion_10,std_9_21_dispersion_15,std_9_21_50_dispersion_15,std_9_21_dispersion_20,std_9_21_50_dispersion_20
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""21STCENMGM""",2024-10-24,113.16,113.16,113.16,113.16,null,null,null,null,null,null,113.16,113.16,null,null,false,0.0,0.0,0.0,0.0,null,null,null,null,null,0.0,0.0,0.0,NaN,NaN,NaN,null,null,false,0.0,0.0,null,0.0,0.0,null,0.0,null,null,null,null,null,null,null,null
"""21STCENMGM""",2024-10-25,113.16,113.16,110.89,110.89,113.16,113.16,113.16,113.16,null,null,111.9,111.97,null,null,false,2.27,2.27,0.0,0.0,0.0,0.0,-0.0203,-0.0203,null,2.27,0.4934,0.4934,1.0,0.0,0.0,null,null,false,-0.009,-0.0096,null,-0.009,-0.0096,null,0.0107,null,null,null,null,null,null,null,null
"""21STCENMGM""",2024-10-28,108.67,108.67,108.67,108.67,113.16,113.16,110.89,110.89,null,null,110.58,110.76,null,null,false,0.0,0.0,0.0,0.0,-0.0405,-0.0405,-0.0202,-0.0202,null,2.22,0.9457,0.9457,NaN,NaN,NaN,null,null,false,-0.0173,-0.0189,null,-0.0173,-0.0189,null,0.0143,null,null,null,null,null,null,null,null
"""21STCENMGM""",2024-10-29,106.49,106.49,106.49,106.49,108.67,108.67,108.67,108.67,null,null,109.19,109.54,null,null,false,0.0,0.0,0.0,0.0,-0.0203,-0.0203,-0.0203,-0.0203,null,2.18,1.3628,1.3628,NaN,NaN,NaN,null,null,false,-0.0247,-0.0278,null,-0.0247,-0.0278,null,0.0161,null,null,null,null,null,null,null,null
"""21STCENMGM""",2024-10-30,104.36,108.61,104.36,108.61,106.49,106.49,106.49,106.49,null,null,109.02,109.32,null,null,true,4.25,4.25,0.0,0.0,-0.0202,0.0197,-0.0202,0.0197,null,4.25,0.291,0.291,1.0,0.0,0.0,null,null,false,-0.0038,-0.0065,null,-0.0427,-0.0454,null,0.0217,0.6186,0.6186,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""GSPCROP""",2026-04-10,387.15,397.0,387.15,389.85,388.5,406.9,385.1,387.05,null,null,381.57,377.02,null,null,true,9.85,2.7,7.15,0.0,-0.0035,-0.0246,0.0053,0.0072,null,9.95,5.3111,5.3111,0.2741,0.7259,0.0,null,null,false,0.0217,0.034,null,0.0146,0.0269,null,0.0659,6.6798,6.6798,5.7078,5.7078,null,null,null,null
"""GSPCROP""",2026-04-13,385.0,409.3,375.0,390.3,387.15,397.0,387.15,389.85,null,null,383.45,378.79,null,null,true,34.3,5.3,19.0,10.0,-0.0056,0.0305,-0.0319,0.0012,null,34.3,4.7272,4.7272,0.1545,0.5539,0.2915,null,null,false,0.0179,0.0304,null,-0.022,-0.01,null,0.0689,5.8884,5.8884,6.1087,6.1087,null,null,null,null
"""GSPCROP""",2026-04-15,396.35,420.0,392.6,413.3,385.0,409.3,375.0,390.3,null,null,389.76,383.21,null,null,true,27.4,16.95,6.7,3.75,0.0291,0.0258,0.0459,0.0573,null,29.7,12.9204,12.9204,0.6186,0.2445,0.1369,null,null,false,0.0604,0.0785,null,0.0073,0.0245,null,0.0693,6.7075,6.7075,7.3406,7.3406,null,null,null,null
